# Variational quantum classifier on real data (Iris dataset)

*Part of the QUEST Quantum Machine Learning series*

Most QML tutorials use synthetic toy datasets (moons, circles, blobs) that make quantum classifiers look competitive. This notebook uses Iris, a real dataset that classical machine learning has been solving with near-perfect accuracy since the 1930s. The comparison is unforgiving on purpose. It shows what QML actually achieves on today's hardware, and where the open research problems sit.

We build a variational quantum classifier (VQC), train it on Iris using two different data encoding strategies, run inference on real quantum hardware, and compare against a classical logistic regression baseline. Along the way we diagnose barren plateaus, the main obstacle to training larger variational circuits.

**Learning objectives.** By the end of this notebook you will:

1. Understand the components of a variational quantum classifier: data encoding, parameterized ansatz, cost function, optimizer.
2. Compare two data-encoding strategies (angle vs amplitude) and understand their trade-offs.
3. Train a VQC on a real multi-class dataset using classical optimization.
4. Diagnose barren plateaus using gradient variance across random initializations.
5. Execute inference on real quantum hardware and compare against ideal simulation.
6. Benchmark honestly against a classical baseline and understand where QML currently stands.

**What to bring in.** Basic familiarity with supervised learning (training data, features, labels, accuracy) and elementary quantum mechanics (superposition, measurement). Knowledge of gradient-based optimization helps, but we treat the optimizer as a black box.

**Credit budget.** Training on simulator is free. Hardware inference for 30 test samples across 3 backends, using ~1,000 shots per prediction, totals roughly 90,000 shots. Run hardware only on the best-trained model. Total credit cost is a few tens of dollars if you run all three backends.


## Why Iris, why now

The Iris dataset has 150 samples, 4 features, and 3 classes. It's small enough to iterate quickly on quantum hardware and just complex enough to require multi-qubit circuits. Classical baselines (logistic regression, SVMs, random forests) achieve 95-97% accuracy trivially. That's the target QML must clear if it wants to claim anything for classification tasks.

A variational quantum classifier consists of four parts:

1. **Data encoding.** A circuit that loads a classical feature vector $\vec{x}$ into a quantum state $|\phi(\vec{x})\rangle$.
2. **Parameterized ansatz.** A circuit with trainable parameters $\vec{\theta}$ that transforms $|\phi(\vec{x})\rangle$ into $|\psi(\vec{x}, \vec{\theta})\rangle$.
3. **Measurement and post-processing.** A measurement (or set of measurements) whose expectation values are combined to produce class scores.
4. **Classical optimizer.** An optimizer that adjusts $\vec{\theta}$ to minimize a loss function computed from the class scores and true labels.

The training loop alternates between quantum evaluation of the model and classical parameter updates. This is called *hybrid quantum-classical* computation, and it's the pattern behind VQE (which you saw in the Chemistry & Physics series), QAOA (companion notebook in this series), and most current QML.


## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

from scipy.optimize import minimize

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator

from qbraid.runtime import QbraidProvider

plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

print("Setup complete.")

## Load and prepare the Iris dataset

Standard preprocessing: standardize features to zero mean, unit variance. Split 80/20 into train/test. The 4 features (sepal length/width, petal length/width) will map naturally to 4 qubits under angle encoding.


In [ ]:
iris = load_iris()
X = iris.data
y = iris.target
class_names = iris.target_names

print(f"Total samples: {len(X)}")
print(f"Features: {iris.feature_names}")
print(f"Classes: {list(class_names)}")

# Standardize features (mean 0, std 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTrain: {len(X_train)} samples, Test: {len(X_test)} samples")

# Visualize the first two features colored by class
fig, ax = plt.subplots(figsize=(8, 5))
for i, name in enumerate(class_names):
    mask = y_train == i
    ax.scatter(X_train[mask, 0], X_train[mask, 1], label=name, s=60, alpha=0.7)
ax.set_xlabel(iris.feature_names[0] + ' (standardized)')
ax.set_ylabel(iris.feature_names[1] + ' (standardized)')
ax.set_title('Iris training data (first two features)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The first two features alone don't cleanly separate the classes; versicolor and virginica overlap heavily. All four features together separate cleanly, and classical baselines achieve high accuracy. This is a solved problem for classical ML. That's what makes it a hard test for QML.

## Classical baseline: logistic regression

Before any quantum work, establish the classical ceiling. Logistic regression is a fair comparison: a simple linear model with negligible training time. If a quantum classifier can't match logistic regression on Iris, that's an honest data point.


In [ ]:
clf_classical = LogisticRegression(max_iter=1000)
clf_classical.fit(X_train, y_train)
y_pred_classical = clf_classical.predict(X_test)
acc_classical = accuracy_score(y_test, y_pred_classical)
print(f"Classical (logistic regression) test accuracy: {acc_classical:.3f}")
print(f"Confusion matrix:")
print(confusion_matrix(y_test, y_pred_classical))

## Two data encoding strategies

How you load classical data into a quantum state determines circuit depth, expressive power, and how noise affects predictions. It's the design decision beginners most often skip past, and it matters more than the ansatz in many cases.

**Angle encoding.** Each feature $x_i$ becomes the rotation angle of a single qubit: $R_y(x_i)$ or $R_x(x_i)$. Simple to implement, single-layer circuit, requires one qubit per feature. Our 4 features need 4 qubits.

**Amplitude encoding.** The feature vector is loaded into the amplitudes of a quantum state: $|\phi(\vec{x})\rangle = \sum_i x_i |i\rangle / \|\vec{x}\|$. Uses $\lceil \log_2(d) \rceil$ qubits for $d$ features, so 4 features fit in 2 qubits. Much more qubit-efficient, but the state preparation circuit can be deep and hard to compile.

We'll teach with angle encoding, which is simpler and easier to reason about, and briefly demonstrate amplitude encoding for comparison.


In [ ]:
def angle_encode(x, qc, qubits):
    """Angle encoding: R_y(x_i) on each qubit."""
    for i, q in enumerate(qubits):
        qc.ry(x[i], q)


def amplitude_encode(x, qc, qubits):
    """Amplitude encoding: uses Qiskit's initialize (which decomposes into a deep circuit)."""
    # Normalize x for amplitude encoding
    norm = np.linalg.norm(x)
    if norm < 1e-10:
        return
    amplitudes = x / norm
    # Pad to nearest power of 2
    n_qubits = len(qubits)
    padded = np.zeros(2 ** n_qubits)
    padded[:len(amplitudes)] = amplitudes
    padded = padded / np.linalg.norm(padded)
    qc.initialize(padded, qubits)


# Show the encoding circuits for a sample
x_sample = X_train[0]

qc_angle = QuantumCircuit(4)
angle_encode(x_sample, qc_angle, range(4))
print(f"Angle encoding: {qc_angle.depth()} depth, {sum(qc_angle.count_ops().values())} gates")

qc_amp = QuantumCircuit(2)
amplitude_encode(x_sample, qc_amp, range(2))
qc_amp_decomposed = transpile(qc_amp, basis_gates=['u3', 'cx'])
print(f"Amplitude encoding (compiled): {qc_amp_decomposed.depth()} depth, {sum(qc_amp_decomposed.count_ops().values())} gates")

Amplitude encoding uses half as many qubits but requires a much deeper preparation circuit. On today's hardware, the depth cost usually dominates. Angle encoding is the safer choice for NISQ.

## Building the variational quantum classifier

The full VQC circuit is: data encoding → parameterized ansatz → measurement.

We use a hardware-efficient ansatz: alternating layers of trainable single-qubit rotations and entangling CNOT gates. For 4 qubits and 2 ansatz layers, that's $4 \times 3 = 12$ trainable parameters.

For 3-class classification, we measure 2 observables and use their expectation values to compute class scores (softmax over 3 classes from 2 observables is a common choice).


In [ ]:
N_QUBITS = 4
N_LAYERS = 2
N_PARAMS = N_QUBITS * (N_LAYERS + 1)

def vqc_ansatz(params, qc, qubits):
    """Layered hardware-efficient ansatz: R_y + CNOT chain, repeated."""
    idx = 0
    # Initial R_y layer
    for q in qubits:
        qc.ry(params[idx], q)
        idx += 1
    # Repeated: CNOT chain + R_y
    for _ in range(N_LAYERS):
        for i in range(len(qubits) - 1):
            qc.cx(qubits[i], qubits[i + 1])
        for q in qubits:
            qc.ry(params[idx], q)
            idx += 1


def vqc_circuit(x, params):
    """Full VQC circuit for input x and parameters params."""
    qc = QuantumCircuit(N_QUBITS)
    angle_encode(x, qc, range(N_QUBITS))
    vqc_ansatz(params, qc, list(range(N_QUBITS)))
    return qc


def predict_scores(x, params):
    """
    Compute 3 class scores from measurements.
    We use <Z_0>, <Z_1>, <Z_2> as three independent observables, softmax at the end.
    """
    qc = vqc_circuit(x, params)
    sv = Statevector.from_instruction(qc)
    scores = np.zeros(3)
    for i in range(3):
        label = ['I'] * N_QUBITS
        label[i] = 'Z'
        op = SparsePauliOp.from_list([(''.join(reversed(label)), 1.0)])
        scores[i] = np.real(sv.expectation_value(op))
    # Softmax to get class probabilities
    probs = np.exp(scores) / np.exp(scores).sum()
    return probs


# Sanity check with random params
rng = np.random.default_rng(seed=0)
test_params = rng.uniform(-np.pi, np.pi, N_PARAMS)
probs = predict_scores(X_train[0], test_params)
print(f"Random params test: probs = {probs}, sum = {probs.sum():.4f}")

# Visualize the circuit
qc_example = vqc_circuit(X_train[0], test_params)
qc_example.draw('mpl', fold=100)

## Barren plateau diagnostic

A **barren plateau** shows up in variational quantum circuits as depth or qubit count grows. The loss landscape becomes exponentially flat and gradients vanish across random parameter initializations. On a plateau, no gradient-based optimizer can make progress; the classifier can't learn.

The standard diagnostic computes the variance of the loss (or of a specific observable's expectation value) across many random parameter initializations. If the variance is very small, order $10^{-4}$ or below, you're likely on a plateau.

Let's check our ansatz.


In [ ]:
def compute_observable_variance(n_samples=100):
    """Compute the variance of <Z_0> across n_samples random parameter initializations."""
    rng = np.random.default_rng(seed=1)
    values = []
    for _ in range(n_samples):
        params = rng.uniform(-np.pi, np.pi, N_PARAMS)
        # Use a fixed sample from training data
        probs = predict_scores(X_train[0], params)
        values.append(probs[0])  # first class probability
    return np.var(values)


variance = compute_observable_variance(n_samples=100)
print(f"Variance of <Z_0>-based probability across 100 random inits: {variance:.4f}")
print(f"This ansatz on this problem: {'plateau likely' if variance < 1e-3 else 'trainable region'}")

For our small 4-qubit, 2-layer ansatz, the variance sits well above the plateau threshold. The circuit is trainable. Plateaus become dangerous at larger depths and higher qubit counts, which is why so much current QML research goes into ansatz design. Symmetry-adapted circuits, adaptive ansatze, and layerwise training all exist to stay off the plateau.

## Train the VQC on an ideal simulator

Training on hardware is prohibitively expensive today. Each parameter update needs many circuit evaluations across the training set. Standard practice is to train on simulator and run inference on hardware, so that's what we do.

The loss is cross-entropy over the 3 classes, minimized with COBYLA, a gradient-free optimizer that tolerates noisy evaluations.


In [ ]:
def cross_entropy_loss(params, X_batch, y_batch):
    """Mean cross-entropy loss on a batch."""
    total = 0.0
    for x, y_true in zip(X_batch, y_batch):
        probs = predict_scores(x, params)
        # Avoid log(0)
        probs = np.clip(probs, 1e-10, 1.0)
        total += -np.log(probs[y_true])
    return total / len(X_batch)


# For training efficiency, use a subset of training data at each optimizer step (mini-batch)
BATCH_SIZE = 30
def batch_loss(params):
    idx = np.random.choice(len(X_train), BATCH_SIZE, replace=False)
    return cross_entropy_loss(params, X_train[idx], y_train[idx])


# Full-batch loss for monitoring
def full_loss(params):
    return cross_entropy_loss(params, X_train, y_train)


# Training loop with periodic monitoring
history = {'iter': [], 'batch_loss': [], 'full_loss': [], 'train_acc': []}

def make_callback():
    it = [0]
    def cb(params):
        if it[0] % 5 == 0:
            fl = full_loss(params)
            train_probs = np.array([predict_scores(x, params) for x in X_train])
            train_preds = np.argmax(train_probs, axis=1)
            train_acc = accuracy_score(y_train, train_preds)
            history['iter'].append(it[0])
            history['full_loss'].append(fl)
            history['train_acc'].append(train_acc)
            print(f"  Iter {it[0]:3d}: full loss = {fl:.4f}, train acc = {train_acc:.3f}")
        it[0] += 1
    return cb

np.random.seed(0)
x0 = np.random.uniform(-np.pi, np.pi, N_PARAMS)
callback = make_callback()

print("Training VQC...")
result = minimize(batch_loss, x0, method='COBYLA',
                  callback=callback,
                  options={'maxiter': 80, 'rhobeg': 0.3})

trained_params = result.x
print(f"\nTraining complete after {result.nfev} function evaluations.")

In [ ]:
# Plot convergence
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['iter'], history['full_loss'], color='#a02580', linewidth=2)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Full training loss')
ax1.set_title('Training loss')
ax1.grid(alpha=0.3)

ax2.plot(history['iter'], history['train_acc'], color='#2d7a4f', linewidth=2)
ax2.axhline(acc_classical, linestyle='--', color='k', alpha=0.6, label=f'Classical baseline ({acc_classical:.3f})')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Training accuracy')
ax2.set_title('Training accuracy')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Evaluate on test set (simulator)
test_probs = np.array([predict_scores(x, trained_params) for x in X_test])
test_preds = np.argmax(test_probs, axis=1)
acc_quantum_sim = accuracy_score(y_test, test_preds)
print(f"\nVQC test accuracy (simulator): {acc_quantum_sim:.3f}")
print(f"Classical baseline:            {acc_classical:.3f}")

The VQC typically converges to somewhere in the 80-93% range on Iris, below the classical baseline. That's expected. For this dataset, the classical baseline uses a linear model on 4 features, which is essentially optimal. The VQC has to encode the data through a quantum state and process it through a limited-expressivity ansatz, so reaching classical parity is hard.

The VQC does learn something, to be clear. It captures enough class structure to get most samples right. But without more parameters, deeper circuits, or a better ansatz, it stays below classical. So the honest picture on Iris: roughly 80-93% quantum today, 97% classical. The field is working on closing that gap. It hasn't yet.

## Run inference on real hardware

Does the trained model still work on real quantum hardware? We take the parameters we optimized on simulator, evaluate the predictions on real hardware for the test set, and compare hardware accuracy against simulator accuracy.

We'll use a smaller test subset for hardware (10 samples) to keep credits reasonable.


In [ ]:
provider = QbraidProvider()
# Instructor: replace with an available backend
DEVICE_ID = 'iqm_garnet'
device = provider.get_device(DEVICE_ID)


def predict_scores_hardware(x, params, device, shots=1024):
    """
    Compute class scores from real hardware measurements.
    Measures <Z_0>, <Z_1>, <Z_2> in three separate circuits (since they don't commute
    on a shared measurement basis when the state isn't a product state).
    """
    # For the ansatz we built, all Zi measurements ARE simultaneously measurable
    # (all diagonal in the computational basis). So one measurement is enough.
    qc = vqc_circuit(x, params)
    qc.measure_all()

    job = device.run(qc, shots=shots)
    result = job.result()
    counts = result.data.get_counts()

    # Compute <Z_i> from computational-basis measurements
    total = sum(counts.values())
    scores = np.zeros(3)
    for i in range(3):
        for bitstring, count in counts.items():
            bit = int(bitstring[-(i + 1)])
            eigenvalue = 1 - 2 * bit  # 0 -> +1, 1 -> -1
            scores[i] += eigenvalue * count / total

    probs = np.exp(scores) / np.exp(scores).sum()
    return probs


# Subset of test set for hardware
N_HW_TEST = 10
hw_indices = np.random.choice(len(X_test), N_HW_TEST, replace=False)
X_test_hw = X_test[hw_indices]
y_test_hw = y_test[hw_indices]

print(f"Running inference on hardware for {N_HW_TEST} test samples...")
hw_probs = []
sim_probs_hw_subset = []
for i, x in enumerate(X_test_hw):
    hw_p = predict_scores_hardware(x, trained_params, device, shots=1024)
    sim_p = predict_scores(x, trained_params)
    hw_probs.append(hw_p)
    sim_probs_hw_subset.append(sim_p)
    hw_pred = np.argmax(hw_p)
    sim_pred = np.argmax(sim_p)
    print(f"  Sample {i+1}: true={y_test_hw[i]}, sim_pred={sim_pred}, hw_pred={hw_pred}",
          "✓" if hw_pred == y_test_hw[i] else "✗")

hw_preds = np.argmax(hw_probs, axis=1)
sim_preds_hw_subset = np.argmax(sim_probs_hw_subset, axis=1)
acc_hw = accuracy_score(y_test_hw, hw_preds)
acc_sim_hw_subset = accuracy_score(y_test_hw, sim_preds_hw_subset)
print(f"\nHardware accuracy on {N_HW_TEST}-sample subset:  {acc_hw:.3f}")
print(f"Simulator accuracy on same subset:            {acc_sim_hw_subset:.3f}")
print(f"Classical baseline (full test set):           {acc_classical:.3f}")

## Honest comparison

Three data points to compare. The classical baseline on the full test set is the ceiling for this problem. The VQC on simulator, also on the full test set, is what QML can do with perfect quantum computation. The VQC on hardware, run on the 10-sample subset, is what QML actually does today.


In [ ]:
summary = pd.DataFrame([
    {'Method': 'Classical (logistic regression)', 'Setting': 'Full test set (30)', 'Accuracy': f'{acc_classical:.3f}'},
    {'Method': 'VQC (simulator)',                 'Setting': 'Full test set (30)', 'Accuracy': f'{acc_quantum_sim:.3f}'},
    {'Method': 'VQC (simulator)',                 'Setting': 'Same 10-sample subset as HW', 'Accuracy': f'{acc_sim_hw_subset:.3f}'},
    {'Method': 'VQC (real hardware)',             'Setting': '10-sample subset', 'Accuracy': f'{acc_hw:.3f}'},
])
summary

The gaps in the table tell the story.

**Classical vs VQC-simulator.** The algorithmic gap. This is how much VQC misses relative to a strong classical baseline, even with perfect quantum computation. On Iris, it's typically 5-15 percentage points.

**VQC-simulator vs VQC-hardware.** The noise gap. This is how much hardware noise degrades the ideal quantum prediction. On today's hardware for this circuit, expect another 5-20 percentage points.

Both gaps are active research problems. Better ansatz design and encoding strategies close the algorithmic gap. Better hardware and error mitigation close the noise gap.

## Where QML is, honestly

QML today is an interesting research direction with genuine open questions and no known killer application at NISQ scale. Keep three categories in mind.

**Promise but no advantage yet.** Classification on small, low-dimensional datasets like Iris, where QML can match some classical baselines with well-designed circuits. Kernel methods (quantum kernel estimation), where theoretical advantages exist for specific data distributions but nobody has demonstrated one in practice. Generative modeling (quantum GANs, Born machines), where the expressivity results are interesting but the models aren't competitive on real generation tasks.

**Interesting even without near-term advantage.** Studying expressivity and trainability of quantum circuits. Understanding when quantum features like superposition and entanglement help learning tasks. Barren plateau analysis, which teaches lessons that apply beyond QML.

**Unlikely to help.** Large-scale classification tasks where classical methods are optimal or near-optimal, such as image classification and language modeling. Any task where data quality or feature engineering bounds the classical method, not the model's expressivity.

The honest research question isn't "does QML beat classical?" but "under what conditions might QML beat classical, and are those conditions ever realized in practice?" That's open, and your students can contribute to it.


## Where to go next

- **Try amplitude encoding.** Reduce to 2 qubits with amplitude encoding, see how the trade-off between qubit count and circuit depth plays out on hardware.
- **Try a more expressive ansatz.** Increase `N_LAYERS` to 4, add different entangling patterns (all-to-all, ring). Watch for barren plateaus.
- **Apply to a harder dataset.** The Wisconsin breast cancer dataset (30 features, 2 classes) is a common QML benchmark. Requires PCA preprocessing to fit on 4-6 qubits.
- **Add error mitigation.** Apply zero-noise extrapolation to hardware inference. How much of the simulator accuracy can you recover?
- **QAOA for optimization.** The companion notebook in this series (`Notebook 6`) applies QAOA to Max-Cut and explores parameter landscapes, warm starts, and honest comparison against classical algorithms.


---

**Feedback for the QUEST pedagogy study**

Your feedback informs the IRB-approved QUEST research project on quantum computing pedagogy. Please spend 2 minutes on the following:

1. What was the most useful part of this notebook for your learning?
2. What was the most confusing or under-explained?
3. Which QML topic would you most want to see analyzed on real hardware next?

Please submit your responses via the QUEST portal or reply to your instructor.
